In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Read the data

In [ ]:
train= pd.read_csv("/kaggle/input/playground-series-s6e1/train.csv") 

test_org = pd.read_csv("/kaggle/input/playground-series-s6e1/test.csv") 
test=test_org.copy()

def drop_val(data):
        data.drop(columns=["id"],inplace=True)

drop_val(train)
drop_val(test)

# Encoding data

In [ ]:
def encode_age (data):
    labels=[0, 1, 2, 3]
    intervals=[16, 19, 21, 22,24]
    
    data['age'] = pd.cut(data['age'], bins=intervals, labels=labels).astype(int)

encode_age(train)
encode_age(test)

In [ ]:
encode_columns=['course_b.com','course_b.sc',
       'course_b.tech', 'course_ba', 'course_bba', 'course_bca',
       'course_diploma', 'study_method_coaching', 'study_method_group study',
       'study_method_mixed', 'study_method_online videos',
       'study_method_self-study','goodstu']


def data_create(data):
    data["total_effort"] = (data["study_hours"] * data["class_attendance"])/10
    data["sleepquality"] = ((data["sleep_hours"]>=7) & (data["sleep_quality"]==2))
    data["goodstu"] = ((data["class_attendance"] > 80) & (data["study_hours"] > 5)).astype(int)

    return data


def one_encode (data):
    data = pd.get_dummies(data, columns=["course","study_method"])
    
    data["internet_access"] = data["internet_access"].apply({"yes": 1, "no": 0}.get)
    data["gender"] = data["gender"].apply({"female": 1, "male": 0, "other":2}.get)
    data["sleep_quality"]=data["sleep_quality"].apply({"poor":0,"average":1,"good":2}.get)
    data["facility_rating"]=data["facility_rating"].map({"low":0,"medium":1,"high":2}.get)
    data["exam_difficulty"]=data["exam_difficulty"].map({"easy":0,"moderate":1,"hard":2}.get)
    
    data[encode_columns] = data[encode_columns].map({True: 1, False: 0}.get)
    return data


data_create(train)
data_create(test)

train = one_encode(train)
test  = one_encode(test)

In [ ]:
df= train.copy()

# Train

In [ ]:
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor

# K-Fold settings
kf = KFold(n_splits=10, shuffle=True, random_state=42)

# Preperation of X and Y
X = df.drop(columns="exam_score", axis=1)
y = df["exam_score"]


# Empty lists for storing the results 
cat_oof_preds = np.zeros(len(X))       
cat_test_preds = np.zeros(len(test))   
cat_scores = []


# Resetting indexes
X = X.reset_index(drop=True)
y = y.reset_index(drop=True)

cat_features_indices = [] 


print("K-Fold Training \n")

cb_params = {
    'bagging_temperature': 0.006952130531190703,
    'border_count': 254,
    'depth': 6,
    'iterations': 1148,
    'l2_leaf_reg': 5,
    'learning_rate': 0.1484872065780541,
    'random_strength': 3.6941233379852148
}


for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    
    # DATA SPLITTING
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # --- B. SCALING (Only for Linear Model) ---
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr) # Ensure columns are numeric!
    X_val_scaled = scaler.transform(X_val)
    
    # Preprocessing test set for Kaggle submission:
    # Linear model uses 'kagtest_scaled', CatBoost uses the original 'test' dataframe.
    kagtest_scaled = scaler.transform(test) 
    
    # --- C. MODEL 1: LINEAR REGRESSION ---
    lin_model = LinearRegression()
    lin_model.fit(X_tr_scaled, y_tr)
    
    # Linear Predictions
    p_lin_train = lin_model.predict(X_tr_scaled) 
    p_lin_val = lin_model.predict(X_val_scaled)   
    p_lin_test = lin_model.predict(kagtest_scaled)
    
    # --- D. RESIDUAL CALCULATION (CRITICAL SECTION) ---
    # CatBoost will learn the residuals: Actual Value - Linear Prediction
    residuals_tr = y_tr - p_lin_train
    
    # Calculate residuals for validation as well:
    # CatBoost aims to predict 'residuals', not the 'exam_score' directly.
    residuals_val = y_val - p_lin_val 

    # --- E. MODEL 2: CATBOOST (Training on Residuals) ---
    model = CatBoostRegressor(
        **cb_params,
        loss_function='RMSE',
        verbose=False,
        allow_writing_files=False
    )

    model.fit(
        X_tr, residuals_tr,               # Train: X and Training Residuals
        cat_features=cat_features_indices, # Categorical columns if any
        eval_set=(X_val, residuals_val),   # Validation target is residuals_val
        early_stopping_rounds=50
    )
    
    # Residual Predictions (CatBoost uses original X_val and test df)
    p_resid_val = model.predict(X_val)
    p_resid_test = model.predict(test)
    
    # --- F. ENSEMBLING / COMBINATION ---
    # Final Prediction = Linear Prediction + CatBoost Error Correction (Residuals)
    fold_preds_val = p_lin_val + p_resid_val
    fold_preds_test = p_lin_test + p_resid_test
    
    # --- G. STORING RESULTS ---
    cat_oof_preds[val_idx] = fold_preds_val
    cat_test_preds += fold_preds_test
    
    # Calculate Score
    fold_rmse = np.sqrt(mean_squared_error(y_val, fold_preds_val))
    cat_scores.append(fold_rmse)
    
    print(f"Fold {fold+1} RMSE: {fold_rmse:.4f}")

# --- FINAL RESULTS ---
print(f"\nAverage RMSE: {np.mean(cat_scores):.4f}")
print(f"Standard Deviation: {np.std(cat_scores):.4f}")

# Average the test predictions (Assuming 10-fold CV)
final_test_predictions = cat_test_preds / 10
final_test_predictions = np.clip(final_test_predictions, 0, 100)

In [ ]:
from sklearn.model_selection import KFold
from xgboost import XGBRegressor

# K-Fold settings
kf = KFold(n_splits=10, shuffle=True, random_state=42)

# Empty arrays for storing results
xgb_oof_preds = np.zeros(len(X))        # Out-of-fold predictions for validation scoring
xgb_test_preds = np.zeros(len(test))    # Test set predictions (Averaged across folds)
xgb_rmse_scores = []

# Resetting indices to avoid alignment issues
X = X.reset_index(drop=True)
y = y.reset_index(drop=True)

print("K-Fold Training Started...\n")

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    
    # --- A. DATA SPLITTING ---
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # --- B. SCALING (IMPORTANT: Fit only on the training set!) ---
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_val_scaled = scaler.transform(X_val)
    # Transform the test data using the current fold's scaler
    kagtest_scaled = scaler.transform(test) 
    
    # --- C. MODEL 1: LINEAR REGRESSION ---
    # (Assuming lin_model was fitted in the previous step or uncomment below)
    # lin_model = LinearRegression()
    # lin_model.fit(X_tr_scaled, y_tr)
    
    # Linear Predictions for residual calculation
    p_lin_train = lin_model.predict(X_tr_scaled) 
    p_lin_val = lin_model.predict(X_val_scaled)
    p_lin_test = lin_model.predict(kagtest_scaled)
    
    # --- D. RESIDUAL CALCULATION ---
    residuals_tr = y_tr - p_lin_train
    
    # --- E. MODEL 2: XGBOOST (Training on Residuals) ---
    xgb_resid = XGBRegressor(
        colsample_bytree=0.7615344684232164,
        gamma=0.03244612355449078,
        learning_rate=0.03039154139343447,
        max_depth=6,
        min_child_weight=7,
        n_estimators=2136,
        reg_alpha=0.2848404943774676,
        reg_lambda=1.1106608420635984,
        subsample=0.8438257335919588,
        random_state=42,
        n_jobs=-1
    )
    
    xgb_resid.fit(X_tr, residuals_tr)
    
    # Residual Predictions
    p_resid_val = xgb_resid.predict(X_val)
    p_resid_test = xgb_resid.predict(test)
    
    # --- F. ENSEMBLING / COMBINATION ---
    # Final Prediction = Linear Prediction + XGBoost Error Correction
    fold_preds_val = p_lin_val + p_resid_val
    fold_preds_test = p_lin_test + p_resid_test
    
    # --- G. STORING RESULTS ---
    # Store validation predictions for this fold
    xgb_oof_preds[val_idx] = fold_preds_val
    # Accumulate test predictions (will be averaged later)
    xgb_test_preds += fold_preds_test
    
    # Calculate Score
    fold_rmse = np.sqrt(mean_squared_error(y_val, fold_preds_val))
    xgb_rmse_scores.append(fold_rmse)
    
    print(f"Fold {fold+1} RMSE: {fold_rmse:.4f}")

# --- FINAL RESULTS ---
print(f"\nAverage RMSE: {np.mean(xgb_rmse_scores):.4f}")
print(f"Standard Deviation: {np.std(xgb_rmse_scores):.4f}")

# Average the test predictions (Assuming 10-fold CV)
final_test_predictions = xgb_test_preds / 10

# Clip results between 0-100 (Exam score logic)
final_test_predictions = np.clip(final_test_predictions, 0, 100)

In [ ]:
# --- ENSEMBLE BLENDING ---

# Assigning weights to each model
weight_cat = 0.45 
weight_xgb = 0.55 

# IMPORTANT CORRECTION: Averaging predictions over 10 folds
# Since we used 10-fold CV and accumulated the results, we divide by 10.
avg_cat_preds = cat_test_preds / 10
avg_xgb_preds = xgb_test_preds / 10

# Final Ensemble Prediction (Calculated using weighted averages)
final_ensemble_preds = (avg_cat_preds * weight_cat) + (avg_xgb_preds * weight_xgb)

# Clipping predictions between 0 and 100
final_ensemble_preds = np.clip(final_ensemble_preds, 0, 100)

print(f"Blending complete! (CatBoost: {weight_cat}, XGBoost: {weight_xgb})")

# --- Submission File Creation ---
submission = pd.DataFrame({
    'id': test_org["id"],
    'exam_score': final_ensemble_preds
})

# Rounding results to 1 decimal place
submission['exam_score'] = submission['exam_score'].round(1)
submission.to_csv('submission.csv', index=False)

print("File ready: submission.csv")
print(submission.head())

In [ ]:
submission